# vigilex -- Notebook 03
## Feature Engineering & Modelling

**Prerequisite:** Notebook 02 has been run -> `data/processed/maude_labeled.parquet` exists.

**Goal of this notebook:**
1. Build meaningful features from raw MAUDE data
2. Train and evaluate a baseline model (Logistic Regression)
3. Train a LightGBM model (the final model for the API)
4. Save the model as `.pkl` for the FastAPI endpoint

---

### Why this order?

Feature engineering before modelling is not just convention -- it is logically necessary:
Raw MAUDE records represent individual reports.
But a recall always concerns a **device** or a **manufacturer** over a **time period**.
We therefore need to aggregate from report level to device/time-window level.
That is the core of the feature engineering here.


---
## 📖 How to Read This Notebook (Beginner Guide)

This is **Notebook 03** — the feature engineering and machine learning modelling step.
It was part of the original plan (before the architecture shifted to the PostgreSQL-native
MedDRA coding pipeline). It shows how you would build a recall prediction model
using classical ML on the MAUDE data.

### Key concepts explained here

**Feature Engineering**
: Transforming raw data into informative inputs for a machine learning model.
  Raw data (dates, free text, product codes) cannot be fed directly into most ML algorithms —
  you need to create numerical representations. Examples here:
  - Severity score from event type categories
  - Rolling window complaint counts (how many reports in the last 30/90 days?)
  - Text features from narrative keywords

**Rolling Window** (time series)
: A sliding time window that computes statistics over a recent period.
  Example: "How many adverse event reports were filed for this device in the last 90 days?"
  This captures trend information that a single report cannot.
  Window sizes used: 30, 90, 180 days.

**LightGBM** (Gradient Boosting)
: A fast, high-performance machine learning algorithm for tabular data.
  It builds an ensemble of decision trees iteratively, where each new tree
  corrects the errors of the previous ones ("gradient boosting").
  Chosen over Random Forest because: faster, handles imbalanced data better,
  built-in support for categorical features.

**Class Imbalance**
: In the recall dataset, only ~5% of devices get recalled. If we trained naively,
  the model would just predict "not recalled" for everything and be 95% accurate —
  but completely useless. `scale_pos_weight` in LightGBM corrects for this by
  giving more weight to the rare positive class.

**Train/Test Split** (time-based)
: We split data by date rather than randomly. Records before 2023 = training set;
  records from 2023 onwards = test set. This simulates real deployment:
  we train on historical data and predict on future data.
  A random split would leak future information into training (data leakage).

**SHAP Values** (model explainability)
: SHapley Additive exPlanations — a method from game theory that computes
  how much each feature contributed to each individual prediction.
  Important for regulatory contexts: we need to be able to explain WHY the model
  flagged a particular device, not just THAT it did.

**ROC-AUC** vs **PR-AUC**:
- ROC-AUC: how well does the model separate positives from negatives overall?
- PR-AUC: how well does it find the rare positives (recalls) without too many false alarms?
  PR-AUC is more informative for imbalanced datasets (where positives are rare).

### Note: Architecture context
This notebook represents the ORIGINAL plan (classical ML recall prediction).
The CURRENT architecture uses a different approach: MedDRA coding (Module 2) +
PRR/ROR signal detection (Module 3 — not yet built). Notebook 02 (recall label join)
will eventually validate whether Module 3 signals could have predicted actual recalls.
---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

ROOT = Path('..') if Path('..').joinpath('data').exists() else Path('.')

df = pd.read_parquet(ROOT / 'data/processed/maude_labeled.parquet')
df['date_received'] = pd.to_datetime(df['date_received'], errors='coerce')
df['date_of_event'] = pd.to_datetime(df['date_of_event'], errors='coerce')

print(f'Records: {len(df):,}')
print(f'Columns: {df.columns.tolist()}')
print(f'\nLabel distribution:')
print(df[['recalled_ever', 'recalled_within_12m']].apply(lambda c: c.value_counts()).T)


## 1. Feature Engineering

### 1a. Severity Score

**Why?** `event_type` and `patient_outcome` are categorical and unequally weighted.
A Death is regulatorily different from a Malfunction with no patient harm.
We build an ordinal score that quantifies the clinical weight of an event.
This score is domain-expert-based -- not ML magic, but medical judgement.

This is important for later explainability: an auditor understands 'Severity Score 4 (Death)'
better than an abstract feature importance value.


In [ ]:
# Severity mapping: clinically motivated, not arbitrary
# Source: FDA MDR 21 CFR 803 classification + EU MDR Annex XIV
EVENT_TYPE_SCORE = {
    'Death':       5,
    'Injury':      3,
    'Malfunction': 1,
    'Other':       1,
    '':            1,
}

OUTCOME_SCORE = {
    'Death':                   5,
    'Life Threatening':        4,
    'Hospitalization':         3,
    'Required Intervention':   2,
    'Disability':              2,
    'Congenital Anomaly':      2,
    'Other':                   1,
    '':                        0,
}

df['event_score']   = df['event_type'].map(EVENT_TYPE_SCORE).fillna(1)
df['outcome_score'] = df['patient_outcome'].map(OUTCOME_SCORE).fillna(0)

# Combined severity score: max(), not sum(), because we want the worst-case
df['severity_score'] = df[['event_score', 'outcome_score']].max(axis=1)

print('Severity Score distribution:')
print(df['severity_score'].value_counts().sort_index())

# Sanity check: Deaths must have score 5
assert df.loc[df['event_type'] == 'Death', 'severity_score'].min() == 5, 'Death score wrong!'
print('Sanity check OK')


### 1b. Rolling Window Features

**Why?** A single MAUDE report says little. But 50 reports for the same device
within 3 months is a strong warning signal -- exactly what triggers recall decisions.

Rolling windows aggregate the reporting history per device over time windows.
This is the core of the early warning system.

**Technical implementation:** `product_code` as device proxy (simpler than brand_name
because spelling is often inconsistent). We use pandas `.rolling()` with
`on='date_received'` for time-based windows.


In [ ]:
# Sort for rolling computation (mandatory)
df = df.sort_values(['product_code', 'date_received']).reset_index(drop=True)

def add_rolling_features(df, group_col, date_col, windows_days):
    """
    Rolling-window features per group.

    Computes for each window size:
    - Report count (complaint volume)
    - Mean severity score
    - Count of severe events (score >= 3)
    """
    df = df.copy()

    for window in windows_days:
        w_str = f'{window}d'

        # Rolling per group
        rolled = (
            df.groupby(group_col, group_keys=False)
            .apply(lambda g: g.set_index(date_col)['severity_score']
                   .rolling(w_str, closed='left')  # closed='left': current report not counted in its own window
                   .agg(['count', 'mean'])
                   .rename(columns={'count': f'reports_{w_str}', 'mean': f'severity_mean_{w_str}'})
                   .reset_index())
        )

        df[f'reports_{w_str}']       = rolled[f'reports_{w_str}'].values
        df[f'severity_mean_{w_str}'] = rolled[f'severity_mean_{w_str}'].values

        # Share of severe events in the window
        severe = (
            df.groupby(group_col, group_keys=False)
            .apply(lambda g: g.set_index(date_col)
                   .assign(is_severe=lambda x: (x['severity_score'] >= 3).astype(int))['is_severe']
                   .rolling(w_str, closed='left').mean()
                   .reset_index())
        )
        df[f'severe_rate_{w_str}'] = severe['is_severe'].values

    return df

# 3 time windows: 90 days (short-term), 180 days (mid-term), 365 days (long-term)
print('Computing rolling-window features (takes ~1-2 minutes)...')
df = add_rolling_features(df, 'product_code', 'date_received', [90, 180, 365])

rolling_cols = [c for c in df.columns if 'reports_' in c or 'severity_mean_' in c or 'severe_rate_' in c]
print(f'\n{len(rolling_cols)} rolling features created:')
print(rolling_cols)
print(f'\nExample (first 5 rows):')
print(df[['product_code', 'date_received'] + rolling_cols].head())


### 1c. Temporal Features

**Why?** Recall risks are not evenly distributed over time.
New devices on the market have different risk dynamics than established products.
Seasonal effects (e.g. more infusion pump reports in winter in clinical settings)
can provide weak signals.

Also: the EU AI Act requires transparency about the temporal limitations of a model.
If we include 'year' as a feature, this must be documented in the model card.


In [ ]:
df['year']  = df['date_received'].dt.year
df['month'] = df['date_received'].dt.month
df['quarter'] = df['date_received'].dt.quarter

# Days since first report for this device (proxy for 'device has been on the market a long time')
first_report = df.groupby('product_code')['date_received'].transform('min')
df['days_since_first_report'] = (df['date_received'] - first_report).dt.days

print('Temporal features:')
print(df[['year', 'month', 'quarter', 'days_since_first_report']].describe())


### 1d. Text Feature from Narrative

**Why?** Narrative texts contain clinical details that structured fields cannot capture.
Full NLP (embeddings, BERT) is overkill for the baseline model --
but a few simple text features give the model a signal from the free text.

For Phase II (RAG), narratives will be used directly as context.
Simple heuristics are sufficient here.


In [ ]:
# Keyword sets: regulatorily relevant terms
HIGH_RISK_KEYWORDS = [
    'death', 'died', 'fatal', 'fatality',
    'overdose', 'occlusion', 'no delivery', 'no insulin',
    'software', 'firmware', 'update', 'defect',
    'recall', 'corrective', 'field safety',
]

def text_features(text):
    if not isinstance(text, str) or len(text) < 5:
        return 0, 0, 0
    text_lower = text.lower()
    n_words    = len(text.split())
    n_keywords = sum(1 for kw in HIGH_RISK_KEYWORDS if kw in text_lower)
    # Normalised: keywords per 100 words
    keyword_density = n_keywords / max(n_words, 1) * 100
    return n_words, n_keywords, keyword_density

text_feat = df['narrative_text'].apply(text_features)
df['narrative_length']          = text_feat.apply(lambda x: x[0])
df['narrative_keyword_count']   = text_feat.apply(lambda x: x[1])
df['narrative_keyword_density'] = text_feat.apply(lambda x: x[2])

print('Text feature statistics:')
print(df[['narrative_length', 'narrative_keyword_count', 'narrative_keyword_density']].describe().round(2))

# Example: report with high keyword density
print('\nExample: report with high keyword density:')
high_kw = df.nlargest(1, 'narrative_keyword_count')
print(high_kw['narrative_text'].values[0][:300])


### 1e. Encode Categorical Features

**Why LabelEncoder instead of OneHot?** LightGBM can work directly with categorical codes
(native categorical support) -- no memory overhead from one-hot columns.
For Logistic Regression, OneHot would be a better fit,
but there we only use numerical features anyway.


In [ ]:
from sklearn.preprocessing import LabelEncoder

cat_cols = ['product_code', 'event_type', 'report_source', 'reporter_country']

le_map = {}
for col in cat_cols:
    le = LabelEncoder()
    df[f'{col}_enc'] = le.fit_transform(df[col].fillna('UNKNOWN').astype(str))
    le_map[col] = le
    print(f'{col}: {le.classes_[:5]}... ({len(le.classes_)} classes)')

print('\nEncoding complete')


## 2. Assemble Feature Set

We work with two targets:
- `recalled_ever` -> simpler, for baseline
- `recalled_within_12m` -> more realistic, for the final model

**Important note on data leakage:**
Rolling-window features are computed with `closed='left'` --
the current report does not count in its own window.
This prevents the model from having the answer already in the features.
This is a common and dangerous mistake with time-series data.


In [ ]:
FEATURE_COLS = [
    # Severity
    'severity_score', 'event_score', 'outcome_score',
    # Rolling Windows
    'reports_90d', 'reports_180d', 'reports_365d',
    'severity_mean_90d', 'severity_mean_180d', 'severity_mean_365d',
    'severe_rate_90d', 'severe_rate_180d', 'severe_rate_365d',
    # Temporal
    'year', 'month', 'quarter', 'days_since_first_report',
    # Text
    'narrative_length', 'narrative_keyword_count', 'narrative_keyword_density',
    # Categorical (encoded)
    'product_code_enc', 'event_type_enc', 'report_source_enc', 'reporter_country_enc',
]

TARGET_SIMPLE  = 'recalled_ever'
TARGET_FINAL   = 'recalled_within_12m'

# Fill missing values with 0 (rolling features have NaN at the start)
df[FEATURE_COLS] = df[FEATURE_COLS].fillna(0)

X = df[FEATURE_COLS]
y_simple = df[TARGET_SIMPLE]
y_final  = df[TARGET_FINAL]

print(f'Feature matrix: {X.shape}')
print(f'\nTarget recalled_ever:       {y_simple.sum():,} positive ({y_simple.mean():.1%})')
print(f'Target recalled_within_12m: {y_final.sum():,} positive ({y_final.mean():.1%})')


## 3. Train/Test Split

**Why a time-based split instead of random?**

With time-series data, a random train/test split is wrong --
it would allow the model to see future information.
In practice this leads to massively overestimated performance.

We split by date: everything before end of 2022 = training, from 2023 onwards = test.
This simulates how the model would be used in production:
trained on historical data, evaluated on new data.
A random split would leak future information into training (data leakage).


In [ ]:
SPLIT_DATE = '2023-01-01'

train_mask = df['date_received'] < SPLIT_DATE
test_mask  = df['date_received'] >= SPLIT_DATE

X_train, X_test = X[train_mask], X[test_mask]
y_train_s, y_test_s = y_simple[train_mask], y_simple[test_mask]
y_train_f, y_test_f = y_final[train_mask],  y_final[test_mask]

print(f'Training: {len(X_train):,} records (before {SPLIT_DATE})')
print(f'Test:     {len(X_test):,} records (from {SPLIT_DATE})')
print(f'\nTrain recall rate (ever):     {y_train_s.mean():.1%}')
print(f'Test  recall rate (ever):     {y_test_s.mean():.1%}')
print(f'Train recall rate (12m):      {y_train_f.mean():.1%}')
print(f'Test  recall rate (12m):      {y_test_f.mean():.1%}')


## 4. Baseline: Logistic Regression on `recalled_ever`

**Why a baseline?**
Before building a complex model, we need a reference value.
If LightGBM is not clearly better than Logistic Regression,
that is either a sign of good feature engineering
or poor data quality -- both are important to know.

**Class imbalance:**
With strongly imbalanced classes (many 0s, few 1s), accuracy is misleading --
a model that always predicts 0 has high accuracy but is completely useless.
We therefore focus on **AUC-ROC** and **Average Precision (PR-AUC)**.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    classification_report, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline

# Pipeline: scaling + model (Logistic Regression requires standardised features)
baseline_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        class_weight='balanced',  # Compensates for class imbalance
        max_iter=500,
        random_state=42
    ))
])

baseline_pipe.fit(X_train, y_train_s)
y_pred_baseline    = baseline_pipe.predict(X_test)
y_proba_baseline   = baseline_pipe.predict_proba(X_test)[:, 1]

auc_baseline = roc_auc_score(y_test_s, y_proba_baseline)
pr_baseline  = average_precision_score(y_test_s, y_proba_baseline)

print(f'=== Baseline: Logistic Regression (recalled_ever) ===')
print(f'AUC-ROC: {auc_baseline:.4f}')
print(f'PR-AUC:  {pr_baseline:.4f}')
print(f'\n{classification_report(y_test_s, y_pred_baseline)}')


## 5. Main Model: LightGBM on `recalled_within_12m`

**Why LightGBM?**
- Very efficient on tabular data (faster than XGBoost)
- Native categorical feature support
- `scale_pos_weight` for class imbalance
- Feature importance out-of-the-box (important for explainability)
- Easily deployable as `.pkl`

**Target:** `recalled_within_12m` is the more realistic label:
'Did this device have a recall within 12 months of this report?'
This is the actual early-warning signal.


In [ ]:
import lightgbm as lgb

# scale_pos_weight: ratio of negatives to positives (compensates for imbalance)
pos = y_train_f.sum()
neg = len(y_train_f) - pos
scale_pw = neg / pos if pos > 0 else 1.0
print(f'scale_pos_weight: {scale_pw:.1f} ({neg:,} negative / {pos:,} positive)')

lgb_model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,
    min_child_samples=50,      # Overfitting protection
    scale_pos_weight=scale_pw,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

# Early stopping: stops if validation AUC does not improve for 20 rounds
# Prevents overfitting without grid search
callbacks = [lgb.early_stopping(stopping_rounds=20, verbose=False),
             lgb.log_evaluation(period=50)]

lgb_model.fit(
    X_train, y_train_f,
    eval_set=[(X_test, y_test_f)],
    eval_metric='auc',
    callbacks=callbacks
)

y_pred_lgb   = lgb_model.predict(X_test)
y_proba_lgb  = lgb_model.predict_proba(X_test)[:, 1]

auc_lgb = roc_auc_score(y_test_f, y_proba_lgb)
pr_lgb  = average_precision_score(y_test_f, y_proba_lgb)

print(f'\n=== LightGBM (recalled_within_12m) ===')
print(f'AUC-ROC: {auc_lgb:.4f}  (Baseline: {auc_baseline:.4f})')
print(f'PR-AUC:  {pr_lgb:.4f}')
print(f'\n{classification_report(y_test_f, y_pred_lgb)}')


## 6. Model Evaluation

### Why ROC + PR curve together?

**ROC curve** shows the trade-off between True Positive Rate and False Positive Rate.
AUC-ROC = 1.0 is perfect, 0.5 = random.

**Precision-Recall curve** is more informative with strong class imbalance:
it shows how many of the predicted recalls are actual recalls (Precision)
vs. how many real recalls we find (Recall).
An early warning system wants high Recall (missing no real recalls),
even if that comes with more false positives.


In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ROC curve
fpr, tpr, _ = roc_curve(y_test_f, y_proba_lgb)
axes[0].plot(fpr, tpr, lw=2, label=f'LightGBM (AUC={auc_lgb:.3f})')
axes[0].plot([0,1],[0,1], 'k--', label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Precision-Recall curve
prec, rec, _ = precision_recall_curve(y_test_f, y_proba_lgb)
axes[1].plot(rec, prec, lw=2, color='orange', label=f'LightGBM (PR-AUC={pr_lgb:.3f})')
baseline_pr = y_test_f.mean()
axes[1].axhline(baseline_pr, ls='--', color='grey', label=f'Random ({baseline_pr:.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()
axes[1].grid(alpha=0.3)

# Feature Importance
importance = pd.Series(
    lgb_model.feature_importances_,
    index=FEATURE_COLS
).sort_values(ascending=True).tail(15)

importance.plot(kind='barh', ax=axes[2], color='steelblue')
axes[2].set_title('Top 15 Feature Importance (LightGBM)')
axes[2].set_xlabel('Importance')

plt.tight_layout()
plt.savefig(ROOT / 'models/evaluation_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('-> models/evaluation_plots.png saved')


## 7. Model Explainability: SHAP Values

**Why SHAP?** Feature importance tells us which features matter -- but not why.
SHAP (SHapley Additive exPlanations) shows for each individual report:
'This report has a high risk score because reports_365d=47 (+0.23 SHAP)
and severity_score=5 (+0.18 SHAP).'

This is regulatorily critical: the EU AI Act (Art. 13) requires transparency
and explainability for high-risk AI systems. SHAP is the standard method for this.


In [ ]:
try:
    import shap

    # SHAP TreeExplainer is optimised for tree models (very fast)
    explainer = shap.TreeExplainer(lgb_model)

    # Sample: 500 test records for SHAP (otherwise too slow)
    X_sample = X_test.sample(min(500, len(X_test)), random_state=42)
    shap_values = explainer.shap_values(X_sample)

    # For binary classification: shap_values[1] = class 1 (recall)
    sv = shap_values[1] if isinstance(shap_values, list) else shap_values

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    plt.sca(axes[0])
    shap.summary_plot(sv, X_sample, feature_names=FEATURE_COLS, show=False, max_display=12)
    axes[0].set_title('SHAP Summary Plot')

    plt.sca(axes[1])
    shap.summary_plot(sv, X_sample, feature_names=FEATURE_COLS,
                      plot_type='bar', show=False, max_display=12)
    axes[1].set_title('SHAP Feature Importance (mean |SHAP|)')

    plt.tight_layout()
    plt.savefig(ROOT / 'models/shap_plots.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('-> models/shap_plots.png saved')

except ImportError:
    print('shap not installed. pip install shap')
    print('SHAP is needed for the Streamlit demo -- please install.')


## 8. Save the Model

We save:
1. The LightGBM model as `.pkl` (for FastAPI)
2. The feature list (so the API endpoint knows which features to expect)
3. The LabelEncoder map (for categorical features in new data)
4. Model metadata as JSON (for transparency / EU AI Act documentation)


In [ ]:
import joblib
import json
from datetime import datetime

model_dir = ROOT / 'models'
model_dir.mkdir(exist_ok=True)

# 1. Model
joblib.dump(lgb_model, model_dir / 'lgb_recall_risk.pkl')
print('-> models/lgb_recall_risk.pkl saved')

# 2. Feature list
joblib.dump(FEATURE_COLS, model_dir / 'feature_cols.pkl')
print('-> models/feature_cols.pkl saved')

# 3. LabelEncoder map
joblib.dump(le_map, model_dir / 'label_encoders.pkl')
print('-> models/label_encoders.pkl saved')

# 4. Model metadata (EU AI Act Art. 13: transparency)
metadata = {
    'model_name':       'vigilex-lgb-v1',
    'model_type':       'LightGBMClassifier',
    'target':           TARGET_FINAL,
    'training_cutoff':  SPLIT_DATE,
    'trained_on':       datetime.now().isoformat(),
    'n_training_records': int(len(X_train)),
    'n_test_records':   int(len(X_test)),
    'auc_roc':          round(auc_lgb, 4),
    'pr_auc':           round(pr_lgb, 4),
    'positive_rate_train': round(float(y_train_f.mean()), 4),
    'positive_rate_test':  round(float(y_test_f.mean()), 4),
    'device_categories':   df['product_code'].unique().tolist(),
    'features':         FEATURE_COLS,
    'risk_classification': 'HIGH_RISK_AI_SYSTEM',  # EU AI Act Annex III
    'intended_use': 'Recall risk prediction for drug delivery medical devices (FDA MAUDE data)',
    'limitations': [
        'Trained only on insulin pump and related device data',
        f'Training data until {SPLIT_DATE}',
        'Recall labels from FDA Recall DB only (US market)',
        'Not validated on EU EUDAMED data'
    ]
}

with open(model_dir / 'model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('-> models/model_metadata.json saved')
print(f'\n=== Final Model Performance ===')
print(f'AUC-ROC: {auc_lgb:.4f}')
print(f'PR-AUC:  {pr_lgb:.4f}')
print(f'Model saved: {model_dir / "lgb_recall_risk.pkl"}')


## Summary & Next Steps

### What did we build?

A LightGBM model that predicts, based on MAUDE reports,
whether a device will have an FDA recall within 12 months.

### What makes the model good?
- Rolling-window features capture patterns over time (not just individual events)
- Severity score is domain-based and explainable
- SHAP values enable per-prediction explanations
- Model metadata documents limitations (EU AI Act compliant)

### Next Steps (Notebook 04 -- FastAPI + Deployment)

1. **FastAPI endpoint** `/predict`: accepts device data, returns risk score + SHAP explanation
2. **RAG pipeline** `/explain`: maps risk score to relevant EU MDR / BfArM articles
3. **Streamlit demo**: simple UI for the presentation
4. **Docker + deployment**: on Hugging Face Spaces or Railway

**Regulatory note:**
Under the EU AI Act Annex III, this model qualifies as a high-risk AI system
(automated decision with potential significant impact on patient safety).
The `model_metadata.json` is a first step toward technical documentation
under EU AI Act Art. 11.
